<a href="https://colab.research.google.com/github/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/blob/main/logging_monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile monitored_pipeline.py
"""Customer segmentation workflow with logging and monitoring."""

import json
import logging
import time
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

FEATURES = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
MIN_SILHOUETTE = 0.25  # warn if clustering quality is below this
MAX_SECONDS = 30       # warn if the workflow is slower than this


def get_logger(log_file: str = "segmentation.log") -> logging.Logger:
    """Create a logger that writes to a file and to the screen."""
    logger = logging.getLogger("segmentation")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(fmt)
        console = logging.StreamHandler()
        console.setFormatter(fmt)
        logger.addHandler(file_handler)
        logger.addHandler(console)
    return logger


def run_monitored_pipeline(path: str, n_clusters: int = 3,
                           log_file: str = "segmentation.log") -> Tuple[pd.DataFrame, Dict]:
    """Run the full workflow, log every step and return monitoring metrics."""
    logger = get_logger(log_file)
    metrics: Dict = {"status": "failed"}
    start = time.perf_counter()
    logger.info("Workflow started (K=%s)", n_clusters)

    try:
        df = pd.read_csv(path)
        if df.empty:
            raise ValueError("Input data is empty")
        metrics["rows_loaded"] = len(df)
        logger.info("Loaded %d rows", len(df))

        missing = [c for c in FEATURES if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")

        df = df.drop_duplicates().dropna(subset=FEATURES)
        metrics["rows_used"] = len(df)
        dropped = metrics["rows_loaded"] - metrics["rows_used"]
        if dropped:
            logger.warning("Dropped %d rows (duplicates or missing values)", dropped)

        if not isinstance(n_clusters, (int, np.integer)) or n_clusters < 2 or n_clusters >= len(df):
            raise ValueError(f"n_clusters={n_clusters} is invalid for {len(df)} rows")

        X = StandardScaler().fit_transform(df[FEATURES])
        logger.info("Features scaled")

        model = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10, random_state=42)
        df["Cluster"] = model.fit_predict(X)
        metrics["inertia"] = round(float(model.inertia_), 3)
        metrics["cluster_sizes"] = {int(k): int(v) for k, v in
                                    df["Cluster"].value_counts().sort_index().items()}
        logger.info("K-Means finished, cluster sizes: %s", metrics["cluster_sizes"])

        if min(metrics["cluster_sizes"].values()) == 1:
            logger.warning("A cluster has only 1 customer")

        score = float(silhouette_score(X, df["Cluster"]))
        metrics["silhouette"] = round(score, 3)
        if score < MIN_SILHOUETTE:
            logger.warning("Low silhouette score: %.3f (below %.2f)", score, MIN_SILHOUETTE)
        else:
            logger.info("Silhouette score: %.3f", score)

        metrics["status"] = "success"
        return df, metrics

    except Exception as exc:
        logger.error("Workflow failed: %s", exc)
        metrics["error"] = str(exc)
        raise

    finally:
        metrics["seconds"] = round(time.perf_counter() - start, 3)
        if metrics["seconds"] > MAX_SECONDS:
            logger.warning("Workflow was slow: %.1fs", metrics["seconds"])
        logger.info("Workflow ended: %s", json.dumps(metrics))
        with open("metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

Writing monitored_pipeline.py


In [2]:
import os
import json
import pandas as pd
import monitored_pipeline as mp

URL = "https://raw.githubusercontent.com/varshayinithangaraju30-lgtm/customer-segmentation-kmeans/main/Mall_Customers.csv"

df, metrics = mp.run_monitored_pipeline(URL, n_clusters=3)
print("\nMetrics:", metrics)


def expect_error(fn, error=ValueError):
    try:
        fn()
    except error:
        return
    raise AssertionError("Expected " + error.__name__)


def test_success_status():
    assert metrics["status"] == "success"

def test_metrics_keys():
    for key in ["rows_loaded", "rows_used", "silhouette", "inertia", "cluster_sizes", "seconds"]:
        assert key in metrics

def test_log_file_created():
    assert os.path.exists("segmentation.log")
    assert "Workflow started" in open("segmentation.log").read()

def test_metrics_file_created():
    saved = json.load(open("metrics.json"))
    assert saved["status"] == "success"

def test_bad_k_logged():
    expect_error(lambda: mp.run_monitored_pipeline(URL, n_clusters=1000))
    assert "Workflow failed" in open("segmentation.log").read()

def test_missing_column_logged():
    pd.DataFrame({"Age": [1, 2, 3]}).to_csv("bad.csv", index=False)
    expect_error(lambda: mp.run_monitored_pipeline("bad.csv"))
    assert "Missing columns" in open("segmentation.log").read()


tests = [test_success_status, test_metrics_keys, test_log_file_created,
         test_metrics_file_created, test_bad_k_logged, test_missing_column_logged]

passed = 0
for t in tests:
    try:
        t()
        print("PASS:", t.__name__)
        passed += 1
    except Exception as e:
        print("FAIL:", t.__name__, "->", e)

print(f"\n{passed}/{len(tests)} tests passed")
print("\n--- segmentation.log (last 10 lines) ---")
print("".join(open("segmentation.log").readlines()[-10:]))

2026-11-01 10:01:34,069 | INFO | Workflow started (K=3)
INFO:segmentation:Workflow started (K=3)
2026-11-01 10:01:34,230 | INFO | Loaded 20 rows
INFO:segmentation:Loaded 20 rows
2026-11-01 10:01:34,249 | INFO | Features scaled
INFO:segmentation:Features scaled
2026-11-01 10:01:34,318 | INFO | K-Means finished, cluster sizes: {0: 7, 1: 7, 2: 6}
INFO:segmentation:K-Means finished, cluster sizes: {0: 7, 1: 7, 2: 6}
2026-11-01 10:01:34,326 | INFO | Silhouette score: 0.389
INFO:segmentation:Silhouette score: 0.389
2026-11-01 10:01:34,329 | INFO | Workflow ended: {"status": "success", "rows_loaded": 20, "rows_used": 20, "inertia": 19.611, "cluster_sizes": {"0": 7, "1": 7, "2": 6}, "silhouette": 0.389, "seconds": 0.26}
INFO:segmentation:Workflow ended: {"status": "success", "rows_loaded": 20, "rows_used": 20, "inertia": 19.611, "cluster_sizes": {"0": 7, "1": 7, "2": 6}, "silhouette": 0.389, "seconds": 0.26}
2026-11-01 10:01:34,336 | INFO | Workflow started (K=1000)
INFO:segmentation:Workflow 


Metrics: {'status': 'success', 'rows_loaded': 20, 'rows_used': 20, 'inertia': 19.611, 'cluster_sizes': {0: 7, 1: 7, 2: 6}, 'silhouette': 0.389, 'seconds': 0.26}
PASS: test_success_status
PASS: test_metrics_keys
PASS: test_log_file_created
PASS: test_metrics_file_created
PASS: test_bad_k_logged
PASS: test_missing_column_logged

6/6 tests passed

--- segmentation.log (last 10 lines) ---
2026-11-01 10:01:34,326 | INFO | Silhouette score: 0.389
2026-11-01 10:01:34,329 | INFO | Workflow ended: {"status": "success", "rows_loaded": 20, "rows_used": 20, "inertia": 19.611, "cluster_sizes": {"0": 7, "1": 7, "2": 6}, "silhouette": 0.389, "seconds": 0.26}
2026-11-01 10:01:34,336 | INFO | Workflow started (K=1000)
2026-11-01 10:01:34,362 | INFO | Loaded 20 rows
2026-11-01 10:01:34,369 | ERROR | Workflow failed: n_clusters=1000 is invalid for 20 rows
2026-11-01 10:01:34,373 | INFO | Workflow ended: {"status": "failed", "rows_loaded": 20, "rows_used": 20, "error": "n_clusters=1000 is invalid for 20 